# 5. Model Development

Train baseline models ด้วย **17 raw input features + preprocessing pipeline** เพื่อให้ artifact ที่ได้สามารถรับ raw feature columns ได้โดยตรง

In [6]:
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline

train_df = pd.read_parquet("../data/processed/train_pre_impute.parquet")
valid_df = pd.read_parquet("../data/processed/valid_pre_impute.parquet")
test_df = pd.read_parquet("../data/processed/test_pre_impute.parquet")

X_train = train_df.drop(columns="Dropout_Risk")
y_train = train_df["Dropout_Risk"]
X_valid = valid_df.drop(columns="Dropout_Risk")
y_valid = valid_df["Dropout_Risk"]
X_test = test_df.drop(columns="Dropout_Risk")
y_test = test_df["Dropout_Risk"]

preprocessor_path = "../models/preprocessing/preprocessor.joblib"
print(X_train.shape, X_valid.shape, X_test.shape)


(560, 17) (120, 17) (120, 17)


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# โหลด preprocessor แยกให้แต่ละ pipeline เพื่อให้แต่ละ model มีของตัวเอง
lr_pipeline = Pipeline([
    ("preprocessor", joblib.load(preprocessor_path)),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

dt_pipeline = Pipeline([
    ("preprocessor", joblib.load(preprocessor_path)),
    ("model", DecisionTreeClassifier(random_state=42)),
])

rf_pipeline = Pipeline([
    ("preprocessor", joblib.load(preprocessor_path)),
    ("model", RandomForestClassifier(random_state=42)),
])

pipelines = {
    "Logistic Regression": lr_pipeline,
    "Decision Tree": dt_pipeline,
    "Random Forest": rf_pipeline,
}


In [8]:
results = []

for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    train_acc = pipeline.score(X_train, y_train)
    valid_acc = pipeline.score(X_valid, y_valid)
    results.append({
        "Model": name,
        "Train Accuracy": train_acc,
        "Valid Accuracy": valid_acc,
        "Gap": train_acc - valid_acc,
    })

results_df = pd.DataFrame(results).sort_values("Valid Accuracy", ascending=False)
results_df

,Model,Train Accuracy,Valid Accuracy,Gap
0,Logistic Regression,0.733929,0.658333,0.075595
2,Random Forest,1.000000,0.608333,0.391667
1,Decision Tree,1.000000,0.591667,0.408333


Decision Tree / Random Forest อาจมี train accuracy สูงมากเมื่อใช้ค่า default เพราะโมเดลมี capacity สูง จึงต้องดู validation gap และทำ hyperparameter tuning ต่อใน Notebook 6

**ยังไม่ใช้ test set เพื่อเลือกโมเดลในขั้นนี้**

In [9]:
from pathlib import Path

Path("../models/baseline").mkdir(parents=True, exist_ok=True)
joblib.dump(pipelines["Logistic Regression"], "../models/baseline/lr_pipeline.joblib")
joblib.dump(pipelines["Decision Tree"], "../models/baseline/dt_pipeline.joblib")
joblib.dump(pipelines["Random Forest"], "../models/baseline/rf_pipeline.joblib")
print("Saved baseline pipelines")

Saved baseline pipelines


In [10]:
# Pipeline รับ raw columns ได้โดยตรง ไม่ต้อง encode/scale เองก่อน predict
sample_raw = X_valid.head(3)
pipelines["Logistic Regression"].predict(sample_raw)

array(['No', 'No', 'No'], dtype=object)